# ViT Transformer Block — 面试版

**难度：** Hard

**面试要点：** 全部手写，不依赖 nn.Linear 和 nn.LayerNorm，用纯张量操作实现

In [ ]:
import torch
import torch.nn as nn
import math

In [ ]:
# ✅ INTERVIEW

class ViTBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_h = d_model // num_heads

        # 面试版: 手动管理所有权重参数
        # QKV 投影权重: (d_model, 3*d_model)
        self.W_qkv = nn.Parameter(torch.randn(d_model, 3 * d_model) * (d_model ** -0.5))
        # 输出投影权重
        self.W_proj = nn.Parameter(torch.randn(d_model, d_model) * (d_model ** -0.5))
        # LayerNorm 的可学习参数 (无 bias 版本的 LN 只有 weight)
        self.ln1_weight = nn.Parameter(torch.ones(d_model))
        self.ln2_weight = nn.Parameter(torch.ones(d_model))
        # MLP 权重
        self.W_fc1 = nn.Parameter(torch.randn(d_model, 4 * d_model) * (d_model ** -0.5))
        self.b_fc1 = nn.Parameter(torch.zeros(4 * d_model))
        self.W_fc2 = nn.Parameter(torch.randn(4 * d_model, d_model) * (d_model ** -0.5))
        self.b_fc2 = nn.Parameter(torch.zeros(d_model))

    def _layer_norm(self, x, weight):
        # ---- 手写 LayerNorm ----
        # 面试高频考点: 手动实现 LayerNorm
        # 公式: (x - mean) / sqrt(var + eps) * gamma
        # dim=-1: 对最后一维（特征维）做归一化
        # keepdim=True: 保持维度以便广播
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)  # 用总体方差
        x_norm = (x - mean) / (var + 1e-6).sqrt()
        return x_norm * weight  # gamma 缩放（beta 偏移省略或设为0）

    def _gelu(self, x):
        return x * 0.5 * (1.0 + torch.erf(x / math.sqrt(2.0)))

    def _mha(self, x):
        B, N, D = x.shape
        # QKV 投影: (B,N,D) @ (D,3D) → (B,N,3D)
        qkv = x @ self.W_qkv
        # 拆分 Q, K, V: 沿最后一维切成3份
        qkv = qkv.reshape(B, N, 3, self.num_heads, self.d_h)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B, H, N, d_h)
        q, k, v = qkv[0], qkv[1], qkv[2]

        scale = self.d_h ** -0.5
        # 注意力分数: (B, H, N, N)
        attn = torch.softmax(q @ k.transpose(-2, -1) * scale, dim=-1)
        # 加权求和: (B, H, N, d_h)
        out = attn @ v
        # 合并多头: (B, N, D)
        out = out.transpose(1, 2).reshape(B, N, D)
        return out @ self.W_proj

    def forward(self, x):
        # Pre-Norm + 残差
        x = x + self._mha(self._layer_norm(x, self.ln1_weight))
        # MLP: Linear → GELU → Linear
        h = self._layer_norm(x, self.ln2_weight)
        h = self._gelu(h @ self.W_fc1 + self.b_fc1)
        x = x + (h @ self.W_fc2 + self.b_fc2)
        return x

In [ ]:
torch.manual_seed(0)
batch, num_patches, d_model, num_heads = 2, 16, 64, 4

block = ViTBlock(d_model, num_heads)
x = torch.randn(batch, num_patches, d_model)
out = block(x)

print("Input shape: ", x.shape)
print("Output shape:", out.shape)
assert out.shape == x.shape

In [ ]:
from torch_judge import check
check("vit_block")

## 📝 核心思路总结

1. **手写 LayerNorm**：`mean/var` + `gamma` 缩放，面试必考
2. **手写 Linear**：用 `nn.Parameter` + `matmul` 替代 `nn.Linear`
3. **GELU 用 erf**：`torch.erf` 是内置函数，不需要近似
4. **权重初始化**：乘以 `d_model ** -0.5` 控制方差